In [ ]:
# Importar las librerías necesarias
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargar el conjunto de datos desde una ruta absoluta
df = pd.read_csv('tmdb_5000_movies.csv')

# 2. Limpieza de los datos
# Mantener solo las columnas necesarias (título y sinopsis)
df = df[['title', 'overview']]
# Eliminar las filas con sinopsis vacías
df = df.dropna(subset=['overview'])

# 3. Preprocesamiento del texto
# Definir una función para limpiar y preprocesar el texto
def preprocess_text(text):
    text = text.lower()  # Convertir a minúsculas
    text = re.sub(r'\W', ' ', text)  # Eliminar caracteres especiales
    text = re.sub(r'\s+', ' ', text)  # Eliminar múltiples espacios en blanco
    tokens = text.split()  # Tokenización
    # Eliminar stopwords utilizando las stopwords de scikit-learn
    filtered_tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return ' '.join(filtered_tokens)

# Aplicar la función de preprocesamiento a las sinopsis
df['processed_overview'] = df['overview'].apply(preprocess_text)

# 4. Cargar embeddings preentrenados de GloVe
model = api.load("glove-wiki-gigaword-100")  # Cargar el modelo GloVe de 100 dimensiones

# 5. Función para obtener embeddings de cada sinopsis
def get_embedding(text, model):
    tokens = text.split()  # Tokenización
    # Obtener embeddings para cada palabra si está en el modelo
    embeddings = [model[word] for word in tokens if word in model]
    # Promediar los embeddings, si no hay embeddings devolver un vector de ceros
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(100)

# Aplicar la función para generar los embeddings de cada sinopsis
df['embedding'] = df['processed_overview'].apply(lambda x: get_embedding(x, model))

# 6. Función para recomendar películas basadas en similitud del coseno
def recommend_movies(movie_title, df):
    # Obtener el embedding de la película dada
    movie_embedding = df[df['title'] == movie_title]['embedding'].values[0].reshape(1, -1)
    # Calcular la similitud del coseno entre la película dada y el resto
    similarities = cosine_similarity(movie_embedding, np.vstack(df['embedding'].values))
    # Añadir la similitud al DataFrame
    df['similarity'] = similarities[0]
    # Ordenar las películas por similitud y devolver las 10 más similares
    recommendations = df.sort_values('similarity', ascending=False)[1:11]  # Excluir la misma película
    return recommendations[['title', 'similarity']]

# 7. Ejemplo de uso
movie_title = 'The Godfather'
recommendations = recommend_movies(movie_title, df)

# Imprimir las películas recomendadas
print(f"Películas recomendadas para '{movie_title}':")
print(recommendations)

Películas recomendadas para 'The Godfather':
                   title  similarity
4143            Rockaway    0.930058
1245          Colombiana    0.928442
1912      Angela's Ashes    0.921628
3496  The Flower of Evil    0.921580
4680         Malevolence    0.919169
2250     The Proposition    0.913812
2147            The Debt    0.913103
1247     City By The Sea    0.911736
1360    There Be Dragons    0.910287
1492          The Reader    0.909946
